# Ensemble Full Run



This notebook is the **full six-crop execution notebook** for the ensemble extension. It recreates the original validation/test splits, loads the saved `*_final.pt` checkpoints, and compares **Hard Voting**, **Soft Voting**, and **Stacking** against the best individual deep learning model for all selected crops.

## Expected Outputs



- `ensemble_comparison_by_crop.csv`

- `ensemble_detailed_results.csv`

- comparison bar chart

- ensemble gain/loss heatmap

- best-ensemble confusion matrix per crop

- classification reports in JSON format

- split indices for reproducibility

In [1]:
from pathlib import Path
import copy
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, auc, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

In [2]:
NOTEBOOK_DIR = Path.cwd().resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == 'ensemble' else NOTEBOOK_DIR



DATA_ROOT = PROJECT_ROOT / 'Dataset' / 'prepro_pbl_dataset'

MODELS_DIR = PROJECT_ROOT / 'checkpoints'

BASE_RESULTS_CSV = PROJECT_ROOT / 'outputs' / 'single_model' / 'csv' / 'collective_training_results.csv'

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'ensemble' / 'full_run'



CROPS_TO_RUN = ['grape', 'maize', 'potato', 'rice', 'tomato', 'wheat']

MODEL_ARCHS = ['MobileNetV2', 'ResNet18', 'EfficientNetB0']

BATCH_SIZE = 32

NUM_WORKERS = 4

RANDOM_STATE = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



MEAN = [0.485, 0.456, 0.406]

STD = [0.229, 0.224, 0.225]



PLOTS_DIR = OUTPUT_DIR / 'plots'

REPORTS_DIR = OUTPUT_DIR / 'reports'

CONFUSION_DIR = OUTPUT_DIR / 'confusion_matrices'

SPLITS_DIR = OUTPUT_DIR / 'split_indices'



for folder in [OUTPUT_DIR, PLOTS_DIR, REPORTS_DIR, CONFUSION_DIR, SPLITS_DIR]:

    folder.mkdir(parents=True, exist_ok=True)



print(f'Device: {DEVICE}')

print(f'Data root: {DATA_ROOT}')

print(f'Models dir: {MODELS_DIR}')

print(f'Output dir: {OUTPUT_DIR}')

Device: cuda
Data root: C:\Users\kashif\Desktop\SML\SML_Lab_Project\Dataset\prepro_pbl_dataset
Models dir: C:\Users\kashif\Desktop\SML\SML_Lab_Project\checkpoints
Output dir: C:\Users\kashif\Desktop\SML\SML_Lab_Project\outputs\ensemble\full_run


In [3]:
eval_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(MEAN, STD),

])





def recreate_crop_splits(crop, data_root=DATA_ROOT, random_state=RANDOM_STATE):

    dataset_path = data_root / crop

    if not dataset_path.exists():

        raise FileNotFoundError(f'Dataset path not found for crop {crop}: {dataset_path}')



    full_dataset = datasets.ImageFolder(str(dataset_path))

    all_idx = list(range(len(full_dataset)))

    all_labels = full_dataset.targets



    train_idx, temp_idx = train_test_split(

        all_idx,

        test_size=0.30,

        stratify=all_labels,

        random_state=random_state,

    )

    val_idx, test_idx = train_test_split(

        temp_idx,

        test_size=0.50,

        stratify=[all_labels[i] for i in temp_idx],

        random_state=random_state,

    )

    return full_dataset, train_idx, val_idx, test_idx





def make_eval_loader(full_dataset, indices, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):

    dataset_copy = copy.deepcopy(full_dataset)

    dataset_copy.transform = eval_transform

    subset = Subset(dataset_copy, indices)

    return DataLoader(

        subset,

        batch_size=batch_size,

        shuffle=False,

        num_workers=num_workers,

        pin_memory=torch.cuda.is_available(),

    )





def prepare_crop_data(crop):

    full_dataset, train_idx, val_idx, test_idx = recreate_crop_splits(crop)

    return {

        'crop': crop,

        'class_names': full_dataset.classes,

        'num_classes': len(full_dataset.classes),

        'train_idx': train_idx,

        'val_idx': val_idx,

        'test_idx': test_idx,

        'val_loader': make_eval_loader(full_dataset, val_idx),

        'test_loader': make_eval_loader(full_dataset, test_idx),

    }





def save_split_indices(crop_data):

    split_frame = pd.DataFrame({

        'split': (['train'] * len(crop_data['train_idx']) + ['val'] * len(crop_data['val_idx']) + ['test'] * len(crop_data['test_idx'])),

        'index': crop_data['train_idx'] + crop_data['val_idx'] + crop_data['test_idx'],

    })

    split_frame.to_csv(SPLITS_DIR / f"{crop_data['crop']}_split_indices.csv", index=False)

In [4]:
def build_model(model_name, num_classes):

    if model_name == 'MobileNetV2':

        model = models.mobilenet_v2(weights=None)

        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.last_channel, num_classes))

        return model



    if model_name == 'ResNet18':

        model = models.resnet18(weights=None)

        model.fc = nn.Linear(model.fc.in_features, num_classes)

        return model



    if model_name == 'EfficientNetB0':

        model = models.efficientnet_b0(weights=None)

        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

        return model



    raise ValueError(f'Unsupported model name: {model_name}')





def safe_torch_load(checkpoint_path, map_location=DEVICE):

    try:

        return torch.load(checkpoint_path, map_location=map_location, weights_only=True)

    except TypeError:

        return torch.load(checkpoint_path, map_location=map_location)





def normalize_state_dict_keys(state_dict):

    if state_dict and all(str(key).startswith('module.') for key in state_dict.keys()):

        return {str(key).replace('module.', '', 1): value for key, value in state_dict.items()}

    return state_dict





def load_saved_model(crop, model_name, num_classes, models_dir=MODELS_DIR, device=DEVICE):

    checkpoint_path = models_dir / f'{crop}_{model_name}_final.pt'

    if not checkpoint_path.exists():

        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')



    model = build_model(model_name, num_classes)

    state_dict = safe_torch_load(checkpoint_path, map_location=device)

    state_dict = normalize_state_dict_keys(state_dict)

    model.load_state_dict(state_dict, strict=True)

    model.to(device)

    model.eval()

    return model

In [5]:
def collect_probabilities(model, loader, device=DEVICE):
    prob_list, pred_list, label_list = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()
            predictions = probabilities.argmax(axis=1)

            prob_list.append(probabilities)
            pred_list.append(predictions)
            label_list.append(labels.numpy())

    return {
        'probabilities': np.concatenate(prob_list, axis=0),
        'predictions': np.concatenate(pred_list, axis=0),
        'labels': np.concatenate(label_list, axis=0),
    }


def validate_alignment(split_name, crop, outputs_by_model):
    reference_labels = next(iter(outputs_by_model.values()))['labels']
    for model_name, payload in outputs_by_model.items():
        if not np.array_equal(reference_labels, payload['labels']):
            raise ValueError(
                f'Label mismatch for crop={crop}, split={split_name}, model={model_name}. '
                'Check split recreation and dataloader consistency.'
            )
    return reference_labels


def hard_voting_outputs(test_outputs, num_classes):
    model_names = list(test_outputs.keys())
    pred_matrix = np.stack([test_outputs[name]['predictions'] for name in model_names], axis=1)
    prob_tensor = np.stack([test_outputs[name]['probabilities'] for name in model_names], axis=1)

    final_predictions = []
    vote_score_rows = []
    for sample_idx in range(pred_matrix.shape[0]):
        votes = pred_matrix[sample_idx]
        vote_counts = np.bincount(votes, minlength=num_classes).astype(np.float32)
        max_votes = vote_counts.max()
        tied_classes = np.flatnonzero(vote_counts == max_votes)
        mean_probabilities = prob_tensor[sample_idx].mean(axis=0)

        if len(tied_classes) == 1:
            chosen_class = int(tied_classes[0])
        else:
            tied_probabilities = mean_probabilities[tied_classes]
            chosen_class = int(tied_classes[int(np.argmax(tied_probabilities))])

        final_predictions.append(chosen_class)
        vote_score_rows.append(vote_counts / len(model_names))

    return {
        'predictions': np.asarray(final_predictions, dtype=np.int64),
        'scores': np.asarray(vote_score_rows, dtype=np.float32),
    }


def soft_voting_outputs(test_outputs):
    averaged_probabilities = np.mean([payload['probabilities'] for payload in test_outputs.values()], axis=0)
    predictions = averaged_probabilities.argmax(axis=1)
    return {
        'predictions': predictions,
        'probabilities': averaged_probabilities,
    }


def make_stacking_features(outputs_by_model):
    return np.concatenate([payload['probabilities'] for payload in outputs_by_model.values()], axis=1)


def stacking_outputs(val_outputs, test_outputs, random_state=RANDOM_STATE):
    x_val = make_stacking_features(val_outputs)
    y_val = next(iter(val_outputs.values()))['labels']
    x_test = make_stacking_features(test_outputs)

    meta_classifier = LogisticRegression(max_iter=2000, random_state=random_state)
    meta_classifier.fit(x_val, y_val)

    return {
        'predictions': meta_classifier.predict(x_test),
        'probabilities': meta_classifier.predict_proba(x_test),
        'meta_classifier': meta_classifier,
    }

In [6]:
def summarise_method(crop, method_name, labels, predictions, class_names):

    label_ids = list(range(len(class_names)))

    accuracy = accuracy_score(labels, predictions) * 100.0

    report = classification_report(

        labels,

        predictions,

        labels=label_ids,

        target_names=class_names,

        output_dict=True,

        zero_division=0,

    )

    matrix = confusion_matrix(labels, predictions, labels=label_ids)

    return {

        'crop': crop,

        'method': method_name,

        'accuracy': accuracy,

        'classification_report': report,

        'confusion_matrix': matrix,

    }





def save_report_json(summary, class_names):

    payload = {

        'crop': summary['crop'],

        'method': summary['method'],

        'accuracy': summary['accuracy'],

        'class_names': class_names,

        'classification_report': summary['classification_report'],

    }

    (REPORTS_DIR / f"{summary['crop']}_{summary['method']}_report.json").write_text(

        json.dumps(payload, indent=2), encoding='utf-8'

    )





def save_confusion_matrix_csv(summary, class_names):

    pd.DataFrame(summary['confusion_matrix'], index=class_names, columns=class_names).to_csv(

        CONFUSION_DIR / f"{summary['crop']}_{summary['method']}_cm.csv"

    )





def plot_confusion_matrix(summary, class_names):

    plt.figure(figsize=(8, 6))

    sns.heatmap(

        summary['confusion_matrix'],

        annot=True,

        fmt='d',

        cmap='Blues',

        xticklabels=class_names,

        yticklabels=class_names,

    )

    plt.title(f"{summary['crop'].title()} - {summary['method']} Confusion Matrix")

    plt.xlabel('Predicted Label')

    plt.ylabel('True Label')

    plt.tight_layout()

    plt.savefig(CONFUSION_DIR / f"{summary['crop']}_{summary['method']}_cm.png", dpi=150, bbox_inches='tight')

    plt.close()





def plot_summary_bar_chart(compact_df):

    melted = compact_df[[

        'crop',

        'best_single_accuracy',

        'hard_voting_accuracy',

        'soft_voting_accuracy',

        'stacking_accuracy',

    ]].melt(id_vars='crop', var_name='method', value_name='accuracy')



    rename_map = {

        'best_single_accuracy': 'Best Single',

        'hard_voting_accuracy': 'Hard Voting',

        'soft_voting_accuracy': 'Soft Voting',

        'stacking_accuracy': 'Stacking',

    }

    melted['method'] = melted['method'].map(rename_map)



    plt.figure(figsize=(12, 6))

    sns.barplot(data=melted, x='crop', y='accuracy', hue='method')

    plt.title('Crop-wise Accuracy Comparison: Best Single vs Ensemble Methods')

    plt.xlabel('Crop')

    plt.ylabel('Accuracy (%)')

    plt.ylim(0, 100)

    plt.tight_layout()

    plt.savefig(PLOTS_DIR / 'ensemble_vs_best_single_bar_chart.png', dpi=150, bbox_inches='tight')

    plt.close()





def plot_delta_heatmap(compact_df):

    delta_df = compact_df[['crop']].copy()

    delta_df['Hard Voting'] = compact_df['hard_voting_accuracy'] - compact_df['best_single_accuracy']

    delta_df['Soft Voting'] = compact_df['soft_voting_accuracy'] - compact_df['best_single_accuracy']

    delta_df['Stacking'] = compact_df['stacking_accuracy'] - compact_df['best_single_accuracy']

    delta_df = delta_df.set_index('crop')



    plt.figure(figsize=(8, 5))

    sns.heatmap(delta_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0)

    plt.title('Ensemble Gain/Loss vs Best Single Model (%)')

    plt.tight_layout()

    plt.savefig(PLOTS_DIR / 'ensemble_delta_vs_best_single_heatmap.png', dpi=150, bbox_inches='tight')

    plt.close()

In [7]:
def maybe_load_existing_single_results():
    if BASE_RESULTS_CSV.exists():
        try:
            return pd.read_csv(BASE_RESULTS_CSV)
        except Exception:
            return None
    return None


compact_rows = []
detailed_rows = []
crop_artifacts = {}
ensemble_display_map = {
    'HardVoting': 'Hard Voting',
    'SoftVoting': 'Soft Voting',
    'Stacking': 'Stacking',
}

for crop in CROPS_TO_RUN:
    crop_data = prepare_crop_data(crop)
    save_split_indices(crop_data)

    val_outputs = {}
    test_outputs = {}

    for model_name in MODEL_ARCHS:
        model = load_saved_model(crop, model_name, crop_data['num_classes'])
        val_outputs[model_name] = collect_probabilities(model, crop_data['val_loader'])
        test_outputs[model_name] = collect_probabilities(model, crop_data['test_loader'])

    test_labels = validate_alignment('test', crop, test_outputs)
    val_labels = validate_alignment('val', crop, val_outputs)

    best_single_model = None
    best_single_accuracy = -1.0
    for model_name, payload in test_outputs.items():
        accuracy = accuracy_score(payload['labels'], payload['predictions']) * 100.0
        detailed_rows.append({'crop': crop, 'method': model_name, 'accuracy': accuracy, 'type': 'single'})
        if accuracy > best_single_accuracy:
            best_single_accuracy = accuracy
            best_single_model = model_name

    hard_output = hard_voting_outputs(test_outputs, crop_data['num_classes'])
    soft_output = soft_voting_outputs(test_outputs)
    stack_output = stacking_outputs(val_outputs, test_outputs)

    ensemble_outputs = {
        'Hard Voting': hard_output,
        'Soft Voting': soft_output,
        'Stacking': stack_output,
    }

    ensemble_summaries = [
        summarise_method(crop, 'HardVoting', test_labels, hard_output['predictions'], crop_data['class_names']),
        summarise_method(crop, 'SoftVoting', test_labels, soft_output['predictions'], crop_data['class_names']),
        summarise_method(crop, 'Stacking', test_labels, stack_output['predictions'], crop_data['class_names']),
    ]

    for summary in ensemble_summaries:
        save_report_json(summary, crop_data['class_names'])
        save_confusion_matrix_csv(summary, crop_data['class_names'])
        detailed_rows.append({'crop': crop, 'method': summary['method'], 'accuracy': summary['accuracy'], 'type': 'ensemble'})

    best_ensemble = max(ensemble_summaries, key=lambda item: item['accuracy'])
    plot_confusion_matrix(best_ensemble, crop_data['class_names'])

    best_final_is_ensemble = best_ensemble['accuracy'] >= best_single_accuracy
    best_final_name = best_ensemble['method'] if best_final_is_ensemble else best_single_model
    best_final_display_name = ensemble_display_map.get(best_final_name, best_final_name)
    best_final_type = 'ensemble' if best_final_is_ensemble else 'single'
    best_final_accuracy = best_ensemble['accuracy'] if best_final_is_ensemble else best_single_accuracy

    compact_rows.append({
        'crop': crop,
        'best_single_model': best_single_model,
        'best_single_accuracy': best_single_accuracy,
        'hard_voting_accuracy': ensemble_summaries[0]['accuracy'],
        'soft_voting_accuracy': ensemble_summaries[1]['accuracy'],
        'stacking_accuracy': ensemble_summaries[2]['accuracy'],
        'best_ensemble_method': best_ensemble['method'],
        'best_ensemble_method_display': ensemble_display_map[best_ensemble['method']],
        'best_ensemble_accuracy': best_ensemble['accuracy'],
        'delta_vs_best_single': best_ensemble['accuracy'] - best_single_accuracy,
        'best_final_name': best_final_name,
        'best_final_display_name': best_final_display_name,
        'best_final_type': best_final_type,
        'best_final_accuracy': best_final_accuracy,
        'final_delta_vs_best_single': best_final_accuracy - best_single_accuracy,
        'tie_break_rule': 'Majority vote; if tied, highest mean softmax probability wins; if still tied, lowest class index wins.',
    })

    crop_artifacts[crop] = {
        'crop': crop,
        'class_names': crop_data['class_names'],
        'num_classes': crop_data['num_classes'],
        'labels': test_labels,
        'val_labels': val_labels,
        'best_single_model': best_single_model,
        'best_ensemble_method': best_ensemble['method'],
        'best_ensemble_method_display': ensemble_display_map[best_ensemble['method']],
        'best_final_name': best_final_name,
        'best_final_display_name': best_final_display_name,
        'best_final_type': best_final_type,
        'single_outputs': test_outputs,
        'ensemble_outputs': ensemble_outputs,
    }

compact_df = pd.DataFrame(compact_rows).sort_values('crop').reset_index(drop=True)
detailed_df = pd.DataFrame(detailed_rows).sort_values(['crop', 'type', 'accuracy'], ascending=[True, True, False]).reset_index(drop=True)
detailed_df['delta_vs_best_single'] = detailed_df.apply(
    lambda row: row['accuracy'] - compact_df.loc[compact_df['crop'] == row['crop'], 'best_single_accuracy'].iloc[0],
    axis=1,
)

existing_results = maybe_load_existing_single_results()
if existing_results is not None and not existing_results.empty:
    existing_best = (
        existing_results
        .sort_values(['crop', 'test_acc', 'inference_ms', 'params_m'], ascending=[True, False, True, True])
        .groupby('crop', as_index=False)
        .first()
        .rename(columns={'model': 'existing_best_single_model', 'test_acc': 'existing_best_single_accuracy'})
    )[['crop', 'existing_best_single_model', 'existing_best_single_accuracy']]
    compact_df = compact_df.merge(existing_best, on='crop', how='left')

compact_df.to_csv(OUTPUT_DIR / 'ensemble_comparison_by_crop.csv', index=False)
detailed_df.to_csv(OUTPUT_DIR / 'ensemble_detailed_results.csv', index=False)
plot_summary_bar_chart(compact_df)
plot_delta_heatmap(compact_df)

compact_df

,crop,best_single_model,best_single_accuracy,hard_voting_accuracy,soft_voting_accuracy,stacking_accuracy,best_ensemble_method,best_ensemble_accuracy,delta_vs_best_single,tie_break_rule,existing_best_single_model,existing_best_single_accuracy
0,grape,ResNet18,99.327052,99.461642,99.596231,99.596231,SoftVoting,99.596231,0.269179,"Majority vote; if tied, highest mean softmax p...",ResNet18,99.327052
1,maize,ResNet18,92.675159,92.834395,92.993631,93.312102,Stacking,93.312102,0.636943,"Majority vote; if tied, highest mean softmax p...",ResNet18,92.675159
2,potato,EfficientNetB0,98.761610,99.071207,99.071207,98.452012,HardVoting,99.071207,0.309598,"Majority vote; if tied, highest mean softmax p...",EfficientNetB0,98.761610
3,rice,ResNet18,88.601036,89.637306,89.378238,91.450777,Stacking,91.450777,2.849741,"Majority vote; if tied, highest mean softmax p...",ResNet18,88.601036
4,tomato,ResNet18,96.821516,97.921760,97.799511,97.799511,HardVoting,97.921760,1.100244,"Majority vote; if tied, highest mean softmax p...",ResNet18,96.821516
5,wheat,ResNet18,88.039632,89.879689,90.304317,91.012031,Stacking,91.012031,2.972399,"Majority vote; if tied, highest mean softmax p...",ResNet18,88.039632


In [8]:
detailed_df

,crop,method,accuracy,type,delta_vs_best_single
0,grape,SoftVoting,99.596231,ensemble,0.269179
1,grape,Stacking,99.596231,ensemble,0.269179
2,grape,HardVoting,99.461642,ensemble,0.134590
3,grape,ResNet18,99.327052,single,0.000000
4,grape,EfficientNetB0,99.327052,single,0.000000
5,grape,MobileNetV2,99.057873,single,-0.269179
6,maize,Stacking,93.312102,ensemble,0.636943
7,maize,SoftVoting,92.993631,ensemble,0.318471
8,maize,HardVoting,92.834395,ensemble,0.159236
9,maize,ResNet18,92.675159,single,0.000000


## Additional Visualizations

Run these cells after the main results cells. They only display figures in the notebook and do not save them.

In [ ]:
viz_df = compact_df[[
    'crop',
    'best_single_accuracy',
    'hard_voting_accuracy',
    'soft_voting_accuracy',
    'stacking_accuracy',
]].melt(id_vars='crop', var_name='method', value_name='accuracy')

method_name_map = {
    'best_single_accuracy': 'Best Single',
    'hard_voting_accuracy': 'Hard Voting',
    'soft_voting_accuracy': 'Soft Voting',
    'stacking_accuracy': 'Stacking',
}
viz_df['method'] = viz_df['method'].map(method_name_map)

plt.figure(figsize=(13, 6))
sns.barplot(data=viz_df, x='crop', y='accuracy', hue='method')
plt.title('Crop-wise Accuracy Comparison: Single Model vs Ensemble Methods')
plt.xlabel('Crop')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.35)
plt.legend(title='Method')
plt.tight_layout()
plt.show()

In [ ]:
delta_plot_df = compact_df[['crop']].copy()
delta_plot_df['Hard Voting'] = compact_df['hard_voting_accuracy'] - compact_df['best_single_accuracy']
delta_plot_df['Soft Voting'] = compact_df['soft_voting_accuracy'] - compact_df['best_single_accuracy']
delta_plot_df['Stacking'] = compact_df['stacking_accuracy'] - compact_df['best_single_accuracy']
delta_plot_df = delta_plot_df.set_index('crop')

plt.figure(figsize=(8, 5))
sns.heatmap(delta_plot_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Ensemble Gain/Loss over Best Single Model (%)')
plt.tight_layout()
plt.show()

In [ ]:
best_ensemble_plot_df = compact_df[['crop', 'best_ensemble_method', 'delta_vs_best_single']].copy()
best_ensemble_plot_df = best_ensemble_plot_df.sort_values('delta_vs_best_single', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=best_ensemble_plot_df, x='crop', y='delta_vs_best_single', hue='best_ensemble_method')
plt.axhline(0, color='black', linewidth=1)
plt.title('Best Ensemble Improvement over Best Single Model')
plt.xlabel('Crop')
plt.ylabel('Accuracy Gain (%)')
plt.grid(axis='y', linestyle='--', alpha=0.35)
plt.tight_layout()
plt.show()

In [ ]:
single_vs_ensemble_summary = compact_df[[
    'crop',
    'best_single_model',
    'best_single_accuracy',
    'best_ensemble_method',
    'best_ensemble_accuracy',
    'delta_vs_best_single',
]]
single_vs_ensemble_summary

## Advanced Visual Comparisons

The next cells add two presentation-friendly views without saving figures:
- a **dumbbell comparison** showing each crop's jump from the best single model to the final chosen approach,
- a **winner map** showing how the best single model compares with the final selected model/method.

In [ ]:
approach_palette = {
    'MobileNetV2': '#F28E2B',
    'ResNet18': '#4E79A7',
    'EfficientNetB0': '#59A14F',
    'Hard Voting': '#E15759',
    'Soft Voting': '#76B7B2',
    'Stacking': '#9C755F',
}

plot_df = compact_df[[
    'crop',
    'best_single_model',
    'best_single_accuracy',
    'best_final_display_name',
    'best_final_accuracy',
    'final_delta_vs_best_single',
]].copy()
plot_df = plot_df.sort_values('best_final_accuracy', ascending=True).reset_index(drop=True)
plot_df['y_pos'] = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(12, 6.5))
ax.hlines(
    y=plot_df['y_pos'],
    xmin=plot_df['best_single_accuracy'],
    xmax=plot_df['best_final_accuracy'],
    color='#B0B7C3',
    linewidth=3,
    alpha=0.85,
)

ax.scatter(
    plot_df['best_single_accuracy'],
    plot_df['y_pos'],
    s=140,
    color='#4E79A7',
    label='Best Single Baseline',
    zorder=3,
)

for row in plot_df.itertuples(index=False):
    final_color = approach_palette.get(row.best_final_display_name, '#333333')
    ax.scatter(
        row.best_final_accuracy,
        row.y_pos,
        s=190,
        color=final_color,
        edgecolor='white',
        linewidth=1.4,
        zorder=4,
    )
    ax.text(
        row.best_final_accuracy + 0.18,
        row.y_pos,
        f"{row.best_final_display_name}  (+{row.final_delta_vs_best_single:.2f}%)",
        va='center',
        fontsize=9,
        color=final_color,
        fontweight='semibold',
    )
    ax.text(
        row.best_single_accuracy - 0.18,
        row.y_pos,
        row.best_single_model,
        va='center',
        ha='right',
        fontsize=8.7,
        color='#4E79A7',
    )

ax.set_yticks(plot_df['y_pos'])
ax.set_yticklabels(plot_df['crop'].str.title())
ax.set_xlabel('Accuracy (%)')
ax.set_ylabel('Crop')
ax.set_title('Best Single Baseline vs Final Chosen Approach (Dumbbell View)')
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.set_xlim(plot_df['best_single_accuracy'].min() - 1.5, min(100, plot_df['best_final_accuracy'].max() + 4.5))
ax.legend(frameon=False, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
stage_df = compact_df[[
    'crop',
    'best_single_model',
    'best_final_display_name',
]].rename(columns={
    'best_single_model': 'Best Single Winner',
    'best_final_display_name': 'Final Chosen Approach',
})

stages = ['Best Single Winner', 'Final Chosen Approach']
fig, ax = plt.subplots(figsize=(10.5, 6.2))

for y_idx, (_, row) in enumerate(stage_df.iterrows()):
    crop = row['crop'].title()
    values = [row[stage] for stage in stages]
    for x_idx, value in enumerate(values):
        rect = plt.Rectangle(
            (x_idx - 0.45, y_idx - 0.42),
            0.9,
            0.84,
            facecolor=approach_palette.get(value, '#CCCCCC'),
            edgecolor='white',
            linewidth=2,
            alpha=0.94,
        )
        ax.add_patch(rect)
        ax.text(
            x_idx,
            y_idx,
            value,
            ha='center',
            va='center',
            fontsize=9,
            color='white',
            fontweight='bold',
            wrap=True,
        )
    ax.text(-0.78, y_idx, crop, ha='right', va='center', fontsize=10.5, fontweight='semibold')

ax.set_xlim(-1.15, len(stages) - 0.35)
ax.set_ylim(-0.65, len(stage_df) - 0.35)
ax.invert_yaxis()
ax.set_xticks(range(len(stages)))
ax.set_xticklabels(stages, fontsize=10, fontweight='semibold')
ax.set_yticks([])
ax.set_title('Winner Map: Single-Model Champion vs Final Selected Approach')
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

## ROC Curves by Crop

The next cell draws a **multiclass micro-average ROC curve** for each crop, comparing:
- the best single model,
- Hard Voting,
- Soft Voting,
- Stacking.

These figures are displayed only and are **not saved**.

In [ ]:
def score_matrix_from_payload(payload):
    if 'probabilities' in payload:
        return payload['probabilities']
    if 'scores' in payload:
        return payload['scores']
    raise KeyError('Payload does not contain probabilities or scores.')


def compute_multiclass_micro_roc(labels, score_matrix, num_classes):
    binarized = label_binarize(labels, classes=np.arange(num_classes))
    fpr, tpr, _ = roc_curve(binarized.ravel(), score_matrix.ravel())
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc


roc_style_map = {
    'Best Single': {'color': '#4E79A7', 'linestyle': '-', 'linewidth': 2.5},
    'Hard Voting': {'color': '#E15759', 'linestyle': '--', 'linewidth': 2.0},
    'Soft Voting': {'color': '#76B7B2', 'linestyle': '-.', 'linewidth': 2.0},
    'Stacking': {'color': '#9C755F', 'linestyle': ':', 'linewidth': 2.6},
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, crop in zip(axes, CROPS_TO_RUN):
    artifact = crop_artifacts[crop]
    labels = artifact['labels']
    num_classes = artifact['num_classes']
    best_single_model = artifact['best_single_model']

    comparison_payloads = {
        'Best Single': artifact['single_outputs'][best_single_model],
        'Hard Voting': artifact['ensemble_outputs']['Hard Voting'],
        'Soft Voting': artifact['ensemble_outputs']['Soft Voting'],
        'Stacking': artifact['ensemble_outputs']['Stacking'],
    }

    for label, payload in comparison_payloads.items():
        score_matrix = score_matrix_from_payload(payload)
        fpr, tpr, roc_auc = compute_multiclass_micro_roc(labels, score_matrix, num_classes)
        style = roc_style_map[label]
        ax.plot(
            fpr,
            tpr,
            label=f"{label} (AUC = {roc_auc:.3f})",
            color=style['color'],
            linestyle=style['linestyle'],
            linewidth=style['linewidth'],
        )

    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    ax.set_title(f"{crop.title()} | Best Single: {best_single_model}")
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.grid(alpha=0.25, linestyle='--')
    ax.legend(fontsize=8, loc='lower right', frameon=False)

for ax in axes[len(CROPS_TO_RUN):]:
    ax.axis('off')

fig.suptitle('Multiclass Micro-Average ROC Curves by Crop', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()